# Endpoints

`Starlette` 包括类`HTTPEndpoint`和`WebSocketEndpoint`，它们提供基于类的视图模式来处理 HTTP 方法调度和 `WebSocket` 会话。

## HTTP端点
该类`HTTPEndpoint`可以用作 `ASGI` 应用程序：



In [ ]:
from starlette.responses import PlainTextResponse
from starlette.endpoints import HTTPEndpoint


class App(HTTPEndpoint):
    async def get(self, request):
        return PlainTextResponse(f"Hello, world!")

如果您使用 Starlette 应用程序实例来处理路由，则可以调度到一个HTTPEndpoint类。请确保调度到类本身，而不是类的实例：

In [ ]:
from starlette.applications import Starlette
from starlette.responses import PlainTextResponse
from starlette.endpoints import HTTPEndpoint
from starlette.routing import Route


class Homepage(HTTPEndpoint):
    async def get(self, request):
        return PlainTextResponse(f"Hello, world!")


class User(HTTPEndpoint):
    async def get(self, request):
        username = request.path_params['username']
        return PlainTextResponse(f"Hello, {username}")

routes = [
    Route("/", Homepage),
    Route("/{username}", User)
]

app = Starlette(routes=routes)from starlette.applications import Starlette
from starlette.responses import PlainTextResponse
from starlette.endpoints import HTTPEndpoint
from starlette.routing import Route


class Homepage(HTTPEndpoint):
    async def get(self, request):
        return PlainTextResponse(f"Hello, world!")


class User(HTTPEndpoint):
    async def get(self, request):
        username = request.path_params['username']
        return PlainTextResponse(f"Hello, {username}")

routes = [
    Route("/", Homepage),
    Route("/{username}", User)
]

app = Starlette(routes=routes)

对于任何未映射到相应处理程序的请求方法，HTTP 端点类将以“405 方法不允许”响应做出响应。

## WebSocket端点
该类WebSocketEndpoint是一个 ASGI 应用程序，它围绕实例的功能提供了一个包装器WebSocket。

ASGI 连接范围可通过端点实例访问，.scope并且具有可选设置的属性encoding，以验证on_receive方法中预期的 websocket 数据。

编码类型有：

- `'json'`
- `'bytes'`
- `'text'`

有三种可覆盖的方法来处理特定的 ASGI websocket 消息类型：

- `async def on_connect(self, websocket, **kwargs)`，用于处理WebSocket连接建立的事件，可以使用await websocket.accept()来完成连接建立。
- `async def on_receive(self, websocket, data)`，用于从WebSocket接收数据。
- `async def on_disconnect(self, websocket, close_code)`，用于处理WebSocket连接终止事件。



`WebSocketEndpoint`比`HTTPEndpoint`要复杂一些。其中要先定义类成员encoding的值，用于指定在WebSocket传输中所使用的数据格式，可取的值有`'json'`、`'bytes'`、`'text'`三种。之后需要实现以下三个方法或者其中的某几个以实现WebSocket通信功能。



In [ ]:
from starlette.endpoints import WebSocketEndpoint


class App(WebSocketEndpoint):
    encoding = 'bytes'

    async def on_connect(self, websocket):
        await websocket.accept()

    async def on_receive(self, websocket, data):
        await websocket.send_bytes(b"Message: " + data)

    async def on_disconnect(self, websocket, close_code):
        pass

也WebSocketEndpoint可以与应用程序类一起使用Starlette：

In [ ]:
import uvicorn
from starlette.applications import Starlette
from starlette.endpoints import WebSocketEndpoint, HTTPEndpoint
from starlette.responses import HTMLResponse
from starlette.routing import Route, WebSocketRoute


html = """
<!DOCTYPE html>
<html>
    <head>
        <title>Chat</title>
    </head>
    <body>
        <h1>WebSocket Chat</h1>
        <form action="" onsubmit="sendMessage(event)">
            <input type="text" id="messageText" autocomplete="off"/>
            <button>Send</button>
        </form>
        <ul id='messages'>
        </ul>
        <script>
            var ws = new WebSocket("ws://localhost:8000/ws");
            ws.onmessage = function(event) {
                var messages = document.getElementById('messages')
                var message = document.createElement('li')
                var content = document.createTextNode(event.data)
                message.appendChild(content)
                messages.appendChild(message)
            };
            function sendMessage(event) {
                var input = document.getElementById("messageText")
                ws.send(input.value)
                input.value = ''
                event.preventDefault()
            }
        </script>
    </body>
</html>
"""

class Homepage(HTTPEndpoint):
    async def get(self, request):
        return HTMLResponse(html)

class Echo(WebSocketEndpoint):
    encoding = "text"

    async def on_receive(self, websocket, data):
        await websocket.send_text(f"Message text was: {data}")

routes = [
    Route("/", Homepage),
    WebSocketRoute("/ws", Echo)
]

app = Starlette(routes=routes)